# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 04 · Embedding

This notebook demonstrates the **Embedding stage** of the RAG pipeline.

The embedding process converts text into high-dimensional vectors, allowing the RAG retrieval process to identify semantic similarities between the user's question and the documents available to answer it.

The project is designed to run on any computer—even without a GPU—as well as on the Streamlit deployment server. To achieve this, we opted for a relatively small Hugging Face embedding model that is nonetheless sufficient for our purposes. Our tests show good performance, with the retrieval component offering the most room for improvement.

The embedding model is loaded using the `create_embedding_model()` function located in `src/github_rag/embedding/embedding_model.py`.

**Notebook objectives**

* Present the selected embedding model.
* Load the model using the `create_embedding_model()` function.
* Generate embeddings for the entire example GitHub repository used throughout the notebook demonstrations.
* Perform some similarity tests to better understand how the embeddings behave.

## Imports and Configuration

In [26]:
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model

You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects, which combines XGBoost and SHAP to build a predictive and explainable system for classifying urban typologies.

In [27]:
REPOSITORY_URL = "https://github.com/davidfernxndez/urban-typology-xai"

## 1. Ingestion -> processing -> chunking

This section apply the Ingestion, processing and chunking phase to get the prepared *LangChain Documents* for Embedding.

In [28]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

Number of documents ingested: 28


In [29]:
# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents before processing: {len(documents_processed)}")

Number of documents before processing: 324


In [30]:
# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents before chunking: {len(documents_chunking)}")

Number of documents before chunking: 985


## 2. Generate Embeddings

The paraphrase-multilingual-MiniLM-L12-v2 model is used to generate the document embeddings. It was selected as a good balance between semantic representation quality, computational efficiency, and portability. Its relatively small size makes it suitable for running locally on CPU-only systems, while also being lightweight enough for deployment in the *Streamlit* application.

The model is a multilingual Sentence-Transformers model based on a 12-layer MiniLM architecture and produces 384-dimensional embeddings. It is designed to capture semantic similarity between sentences and short texts across multiple languages, making it suitable for a GitHub repository RAG system where both the repository content and user queries may contain natural language in different languages.

In [31]:
# Set the HuggingFace embedding model name
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Load the model
embedding_model = create_embedding_model(embedding_model_name)
print(type(embedding_model))

<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>


During the embedding process, only the text content of each document chunk is converted into a vector. The document metadata is not included in the embedding. Instead, it is stored separately in the vector database alongside the corresponding embedding, allowing the metadata to be retrieved together with the relevant document chunks.

In [32]:
# Extract only the text
texts = [document.page_content for document in documents_chunking]

# Generate embeddings
start_time = time.time()
document_embeddings = embedding_model.embed_documents(texts)
print(f"Number of embeddings: {len(document_embeddings)}")
print(f"Embedding dimension: {len(document_embeddings[0])}")
print(f"Embedding generation time: {time.time() - start_time:.2f} seconds")

Number of embeddings: 985
Embedding dimension: 384
Embedding generation time: 28.10 seconds


Now, each document is represented by a 384-dimensional vector that captures the semantic representation of its text in the embedding space.

In [34]:
print("Example of embedding:")
print(document_embeddings[0][:10])

Example of embedding:
[0.12501364946365356, -0.20934517681598663, 0.14126674830913544, 0.26360228657722473, 0.03386077657341957, -0.0891847014427185, -0.43892231583595276, -0.022204851731657982, -0.36138445138931274, 0.19708240032196045]


### 2.1 Similarities experiments

We can run some simple similarity examples to verify that semantically similar texts are mapped to nearby points in the embedding space. To do this, we calculate the cosine similarity between a potential user query and the document embeddings, retrieving the five most similar documents. This provides a very basic test of the RAG retrieval process, intended simply to illustrate the underlying concept.

In [35]:
# User query about the repository
query = "What is the validation strategy used in performance evaluation?"

# Generate the embedding of the query
query_embedding = embedding_model.embed_query(query)

# Calculate similarity with all the embedding documents available
similarities = cosine_similarity(
    [query_embedding],
    document_embeddings
)[0]

# Select the top 5 with highest similarity
top_indices = np.argsort(similarities)[::-1][:5]

for index in top_indices:
    print(f"Similarity: {similarities[index]:.4f}")
    print(f"Source: {documents_chunking[index].metadata.get('source')}")
    print(documents_chunking[index].page_content[:300])
    print("-" * 80)

Similarity: 0.7459
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/4.1_Global_interp_methodology.ipynb
En el protocolo experimental de evaluación de rendimiento, presentado en el *notebook* [3.1_Performance_methodology.ipynb](3.1_Performance_methodology.ipynb), se ha empleado la estrategia *Nested Cross-Validation* con el objetivo de obtener una estimación robusta de la capacidad de generalización de
--------------------------------------------------------------------------------
Similarity: 0.7154
Source: https://github.com/davidfernxndez/urban-typology-xai/blob/main/notebooks/3.1_Performance_methodology.ipynb
## Metodología de evaluación de rendimiento y ejecución de experimentos  
### 📝 Descripción del notebook  
Este *notebook* detalla la metodología aplicada para la evaluación del rendimiento. En primer lugar, se formalizan los modelos seleccionados y las métricas empleadas para analizar su desempeño 
-----------------------------------------------

We can see that related questions have high similarity, whereas different questions have little relation.

In [36]:
queries = [
    "Which metrics are used to measure performance?",
    "Is LightGBM used in the project?",
    "List the evaluation metrics",
]

query_embeddings = np.array([
    embedding_model.embed_query(query)
    for query in queries
])

similarity_matrix = cosine_similarity(query_embeddings)

print(similarity_matrix)

[[1.         0.23634336 0.72787797]
 [0.23634336 1.         0.11455407]
 [0.72787797 0.11455407 1.        ]]
